In [ ]:
# Code is taken from official paper notebook
# https://github.com/lmcinnes/umap_paper_notebook

In [23]:
import numpy as np
import pandas as pd
import sklearn.neighbors
import sklearn.model_selection
import sklearn.datasets
import sklearn.manifold
import sklearn.decomposition

In [24]:
# https://github.com/lmcinnes/umap_paper_notebooks/blob/master/CrossVal%20KNN%20Classifier%20-%20Pendigits.ipynb

In [25]:
def score_embedding(data, labels, k=10, n_neighbors=5, random_state=42):
    model = sklearn.neighbors.KNeighborsClassifier(n_neighbors=n_neighbors)
    cv = sklearn.model_selection.StratifiedKFold(n_splits=k)
    return sklearn.model_selection.cross_val_score(model, data, labels, cv=cv)

In [26]:
def compare_embeddings(embeddings, n_neighbors=5, k=10, random_state=42):
    cross_val_scores = {}
    for alg in embeddings.keys():
        cross_val_scores[alg] = score_embedding(embeddings[alg][['x', 'y']].values, 
                                                embeddings[alg]['labels'].values,
                                                n_neighbors=n_neighbors,
                                                k=k,
                                                random_state=random_state)
    return pd.DataFrame(cross_val_scores).assign(n_neighbors=n_neighbors)

In [27]:
import numpy as np
import pandas as pd
import umap
import sklearn.manifold
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE

digits = load_digits()
data = digits.data
labels = digits.target
# code taken from https://github.com/lmcinnes/umap_paper_notebooks/blob/master/UMAP%20Pen%20Digits.ipynb
fit = umap.UMAP(n_neighbors=15, random_state=42, metric='euclidean')
u = fit.fit_transform(data)
output = pd.DataFrame(u, columns=('x','y'))
output['labels'] = labels
output.to_csv('embedding_pendigits_umap1.csv')

fit_tsne = TSNE(n_jobs=1, random_state=42) # used TSNE from sklearn instead of MulticoreTSNE (as used by the authors) because MulticoreTSNE didnt want to install
u_tsne = fit_tsne.fit_transform(data)
output = pd.DataFrame(u_tsne, columns=('x','y'))
output['labels'] = labels
output.to_csv('embedding_pendigits_tsne1.csv')

pca = PCA(n_components=2)
u_pca = pca.fit_transform(data)
output = pd.DataFrame(u_pca, columns=('x','y'))
output['labels'] = labels
output.to_csv('embedding_pendigits_pca.csv')

u_laplacian = sklearn.manifold.SpectralEmbedding().fit_transform(data)
output = pd.DataFrame(u_laplacian, columns=('x','y'))
output['labels'] = labels
output.to_csv('embedding_pendigits_eigenmaps1.csv')

/home/reinoutvrielink/.local/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [28]:
pendigits = {}
pendigits['tsne'] = pd.read_csv('embedding_pendigits_tsne1.csv', index_col=0)
pendigits['umap'] = pd.read_csv('embedding_pendigits_umap1.csv', index_col=0)
# Skipped creating the LargeVis embeddings myself because the original implementation 
# relies on a legacy C++ codebase that (according to Claude) requires manual compilation and external 
# libraries (like GSL), which frequently fail on modern systems
# That is why we use the pre computed embeddings by the authors
pendigits['lvis'] = pd.read_csv('embedding_pendigits_largevis1.csv', index_col=0)
pendigits['leig'] = pd.read_csv('embedding_pendigits_eigenmaps1.csv', index_col=0)
pendigits['pca'] = pd.read_csv('embedding_pendigits_pca.csv', index_col=0)

In [29]:
df = pd.concat(compare_embeddings(pendigits, n_neighbors=n) for n in (10,20,40,80,160))

In [30]:
df.groupby('n_neighbors', ).mean()

,tsne,umap,lvis,leig,pca
n_neighbors,,,,,
10,0.973852,0.973861,0.962185,0.777405,0.621558
20,0.974398,0.976629,0.969398,0.777967,0.631586
40,0.952716,0.958281,0.949392,0.777952,0.634922
80,0.950493,0.951061,0.943827,0.770161,0.646601
160,0.947145,0.951061,0.913228,0.744578,0.631009


In [31]:
df.groupby('n_neighbors', ).std() * 2

,tsne,umap,lvis,leig,pca
n_neighbors,,,,,
10,0.045687,0.042873,0.064310,0.116826,0.086990
20,0.039329,0.036238,0.051433,0.122755,0.082402
40,0.063579,0.049220,0.077406,0.116089,0.071094
80,0.059834,0.064517,0.081395,0.109518,0.076798
160,0.060982,0.064517,0.085805,0.115847,0.097927
